# 过滤 LangSmith 中 ChatAnthropic 输出与 Agent 关系

目标：
- 仅保留每次 `ChatAnthropic` 调用的关键信息
- 标注每次调用归属到哪个 agent
- 保留每个 agent 的简要输出
- 生成输出到输出的关系（按时间顺序）

输入：`project_traces_task_filter.json`
输出：`project_traces_task_filter.chat_anthropic.filtered.json`

In [ ]:
import json
from pathlib import Path

SCRIPT_DIR = Path.cwd()
INPUT_PATH = SCRIPT_DIR / 'project_traces_task_filter.json'
OUTPUT_PATH = SCRIPT_DIR / 'project_traces_task_filter.chat_anthropic.filtered.json'

if not INPUT_PATH.exists():
    alt = Path('src/workplace/memorysrc/project_traces_task_filter.json')
    if alt.exists():
        INPUT_PATH = alt
        OUTPUT_PATH = alt.parent / 'project_traces_task_filter.chat_anthropic.filtered.json'

raw = json.loads(INPUT_PATH.read_text())
print(f'Loaded traces: {len(raw)}')
print(f'Input path: {INPUT_PATH}')
print(f'Output path: {OUTPUT_PATH}')


In [ ]:
EXCLUDE_AGENT_NAMES = {'LangGraph', 'RunnableSequence'}

def flatten_nodes(root):
    out = []
    def walk(node):
        if not isinstance(node, dict):
            return
        out.append(node)
        for ch in node.get('children') or []:
            walk(ch)
    walk(root)
    return out

def compact_text(v, max_len=3000):
    if v is None:
        return None
    s = v if isinstance(v, str) else json.dumps(v, ensure_ascii=False)
    return s[:max_len] + ' ...[truncated]' if len(s) > max_len else s

def extract_text_blocks(content):
    if isinstance(content, str):
        return content.strip()
    if not isinstance(content, list):
        return None
    parts = []
    for block in content:
        if not isinstance(block, dict):
            continue
        if block.get('type') == 'text':
            t = (block.get('text') or '').strip()
            if t:
                parts.append(t)
    return '\n'.join(parts) if parts else None

def first_non_empty(*vals):
    for v in vals:
        if v is not None and v != '':
            return v
    return None

def extract_input_text(chat_node):
    messages = ((chat_node.get('inputs') or {}).get('messages')) or []
    if not isinstance(messages, list) or not messages:
        return None

    last_msg = messages[-1] if isinstance(messages[-1], dict) else None
    if not last_msg:
        return None

    content = last_msg.get('content')
    return compact_text(extract_text_blocks(content), 1200)

def extract_generation_message(chat_node):
    generations = ((chat_node.get('outputs') or {}).get('generations')) or []
    if not generations or not isinstance(generations, list):
        return {}
    g0 = generations[0] if generations else []
    if not g0 or not isinstance(g0, list):
        return {}
    first = g0[0] if g0 and isinstance(g0[0], dict) else {}
    msg = first.get('message') or {}
    kwargs = (msg.get('kwargs') if isinstance(msg, dict) else {}) or {}

    text = first_non_empty(
        first.get('text'),
        extract_text_blocks(kwargs.get('content'))
    )

    tool_calls = kwargs.get('tool_calls')
    if not isinstance(tool_calls, list):
        tool_calls = []

    response_meta = kwargs.get('response_metadata') or {}
    llm_output = ((chat_node.get('outputs') or {}).get('llm_output')) or {}

    model_name = first_non_empty(
        response_meta.get('model_name'),
        response_meta.get('model'),
        llm_output.get('model_name'),
        llm_output.get('model')
    )

    stop_reason = first_non_empty(
        response_meta.get('stop_reason'),
        llm_output.get('stop_reason')
    )

    return {
        'text': compact_text(text, 3000),
        'tool_calls': [
            {
                'id': tc.get('id'),
                'name': tc.get('name'),
                'args': tc.get('args'),
                'type': tc.get('type'),
            }
            for tc in tool_calls
            if isinstance(tc, dict)
        ],
        'model': model_name,
        'stop_reason': stop_reason,
    }

def extract_agent_output_summary(agent_node):
    outputs = agent_node.get('outputs') or {}
    if not isinstance(outputs, dict):
        return {}

    summary = {}
    if 'final_answer' in outputs:
        summary['final_answer'] = compact_text(outputs.get('final_answer'), 3000)

    out = outputs.get('output')
    if isinstance(out, dict):
        brief = {}
        if 'goto' in out:
            brief['goto'] = out.get('goto')

        update = out.get('update')
        if isinstance(update, dict):
            for k in ('pending_step_response', 'task_failure_report'):
                if k in update and update.get(k):
                    brief[k] = compact_text(update.get(k), 2000)

        if brief:
            summary['output'] = brief

    if agent_node.get('error'):
        summary['error'] = compact_text(agent_node.get('error'), 2000)

    return summary

def find_owner_agent(node, id_to_node, root_name):
    parent_id = node.get('parent_run_id')
    while parent_id:
        p = id_to_node.get(parent_id)
        if not isinstance(p, dict):
            return None

        p_name = p.get('name')
        if p.get('run_type') == 'chain' and p_name and p_name not in EXCLUDE_AGENT_NAMES and p_name != root_name:
            return p

        parent_id = p.get('parent_run_id')

    return None


In [ ]:
filtered = []

for root in raw:
    root_name = root.get('name')
    nodes = flatten_nodes(root)
    id_to_node = {n.get('id'): n for n in nodes if isinstance(n, dict) and n.get('id')}

    chats = []
    for n in nodes:
        if not isinstance(n, dict):
            continue
        if n.get('name') != 'ChatAnthropic':
            continue

        owner = find_owner_agent(n, id_to_node, root_name)
        owner_id = owner.get('id') if owner else None
        owner_name = owner.get('name') if owner else None

        gen = extract_generation_message(n)
        chats.append({
            'chat_run_id': n.get('id'),
            'parent_run_id': n.get('parent_run_id'),
            'agent_run_id': owner_id,
            'agent_name': owner_name,
            'start_time': n.get('start_time'),
            'end_time': n.get('end_time'),
            'input_text': extract_input_text(n),
            'output_text': gen.get('text'),
            'tool_calls': gen.get('tool_calls') or [],
            'model': gen.get('model'),
            'stop_reason': gen.get('stop_reason'),
            'error': compact_text(n.get('error'), 2000) if n.get('error') else None,
            'agent_output': extract_agent_output_summary(owner) if owner else {},
        })

    chats.sort(key=lambda x: ((x.get('start_time') or ''), (x.get('chat_run_id') or '')))

    relations = []
    for i in range(1, len(chats)):
        prev = chats[i - 1]
        cur = chats[i]
        relations.append({
            'from_chat_run_id': prev.get('chat_run_id'),
            'from_agent': prev.get('agent_name'),
            'to_chat_run_id': cur.get('chat_run_id'),
            'to_agent': cur.get('agent_name'),
            'relation': 'next_output',
            'cross_agent': prev.get('agent_name') != cur.get('agent_name'),
        })

    filtered.append({
        'workflow': {
            'trace_id': root.get('trace_id'),
            'workflow_run_id': root.get('id'),
            'workflow_name': root.get('name'),
            'start_time': root.get('start_time'),
            'end_time': root.get('end_time'),
        },
        'chat_anthropic_outputs': chats,
        'relations': relations,
    })

OUTPUT_PATH.write_text(json.dumps(filtered, ensure_ascii=False, indent=2))

print(f'Wrote: {OUTPUT_PATH}')
print(f'Workflow count: {len(filtered)}')
print(f'ChatAnthropic runs: {sum(len(x["chat_anthropic_outputs"]) for x in filtered)}')
print(f'Relation edges: {sum(len(x["relations"]) for x in filtered)}')

# 预览一个样本
sample = filtered[0] if filtered else {}
print(json.dumps(sample, ensure_ascii=False, indent=2)[:3000])


In [ ]:
content= \
[
    {
        'text': '## Reasoning & Plan\n\n* **Analysis:** This task requires finding the intersection of three datasets:\n  1. OECD countries with ≥20% foreign-born population (2023) from Oxford Migration Observatory\n  2. Countries with criminality score increase of ≥+0.2 (2021→2023) from Organised Crime Index\n  3. Countries with resilience score decrease of >0.3 (2021→2023) from Organised Crime Index\n\n* **Plan:**\n  1. Search for OECD countries migration data from Oxford Migration Observatory (2023)\n  2. Search for Organised Crime Index data (2021 and 2023)\n  3. Fetch and extract the relevant data from these sources\n  4. Identify OECD countries with ≥20% foreign-born population\n  5. Among those, find which had criminality +≥0.2 AND resilience decrease >0.3\n  6. Return the country that meets all criteria\n\nLet me start by searching for these data sources.', 
        'type': 'text'
    }, 
    {
        'id': 'toolu_01XQWN5avvcjm7Wyk4iXgQc8',
        'input': {'query': 'OECD countries foreign-born population 2023 Oxford Migration Observatory', 'num_results': 10}, 
        'name': 'web_search', 
        'type': 'tool_use', 
        'caller': {'type': 'direct'}
    }, 
    {
        'id': 'toolu_01QgwMRVvL8cf321KaP3vyts',
        'input': {'query': 'Organised Crime Index 2021 2023 criminality resilience scores', 'num_results': 10}, 
        'name': 'web_search', 
        'type': 'tool_use', 
        'caller': {'type': 'direct'}
    }
] 
additional_kwargs={} 
response_metadata={'id': 'msg_01Mjd7rok8eqxZ3oDz6gySbS', 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 2259, 'output_tokens': 394, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-sonnet-4-5-20250929'} id='run--8f22bb3c-40ae-4812-b4b9-ca04e0414679-0' 
tool_calls= \
[
    {
        'name': 'web_search', 
        'args': {'query': 'OECD countries foreign-born population 2023 Oxford Migration Observatory', 'num_results': 10}, 
        'id': 'toolu_01XQWN5avvcjm7Wyk4iXgQc8', 
        'type': 'tool_call'
    }, 
    {
        'name': 'web_search', 
        'args': {'query': 'Organised Crime Index 2021 2023 criminality resilience scores', 'num_results': 10}, 
        'id': 'toolu_01QgwMRVvL8cf321KaP3vyts', 'type': 'tool_call'
    }
] 
usage_metadata={'input_tokens': 2259, 'output_tokens': 394, 'total_tokens': 2653, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}}
print(content)

SyntaxError: invalid syntax (4129569973.py, line 3)

## only save the trace whose trace_id == 0e8eb6b4-d1b3-45b1-a009-b5de22cb7761

In [1]:
import json

# 读取所有 traces
traces = json.load(open('project_traces.json'))
# 筛选 trace_id 为 0e8eb6b4-d1b3-45b1-a009-b5de22cb7761 的 trace
filtered_traces = [trace for trace in traces if trace['trace_id'] == '0e8eb6b4-d1b3-45b1-a009-b5de22cb7761']
# 保存筛选后的 traces
with open('project_traces_0e8eb6b4-d1b3-45b1-a009-b5de22cb7761.json', 'w') as f:
    json.dump(filtered_traces, f, indent=2)
